# Notebook 04 — Phase 2: HMM Evaluation

Evaluate the trained HMM on the 54 historical landslide records:  
- Per-sequence log-likelihood scores  
- Viterbi state decoding visualisation  
- Leave-one-out type classification accuracy  
- Occurrence probability distribution

In [ ]:
import sys, os, numpy as np
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
import config
import matplotlib.pyplot as plt

## 1. Load Model, Encoder & Data

In [ ]:
from phase2_hmm.hmm_model import load_hmm_model, load_encoder
from phase2_hmm.data_preprocessing import load_excel_data, clean_and_encode, build_observation_sequences

hmm_model = load_hmm_model(config.HMM_MODEL_PATH)
dis_enc   = load_encoder(config.HMM_ENCODER_PATH)

df = load_excel_data(config.EXCEL_FILE)
df_clean, _, _ = clean_and_encode(df)
obs_sequences, lengths = build_observation_sequences(df_clean)

print(f'Sequences loaded: {len(obs_sequences)}')

## 2. Per-Sequence Log-Likelihood

In [ ]:
ll_scores = []
for seq in obs_sequences:
    X = seq.reshape(-1, 1)
    ll = hmm_model.score(X)
    ll_scores.append(ll)

print(f'Mean log-likelihood : {np.mean(ll_scores):.4f}')
print(f'Std  log-likelihood : {np.std(ll_scores):.4f}')
print(f'Min  log-likelihood : {min(ll_scores):.4f}')
print(f'Max  log-likelihood : {max(ll_scores):.4f}')

plt.figure(figsize=(8, 4))
plt.bar(range(len(ll_scores)), ll_scores, color='steelblue', edgecolor='white')
plt.xlabel('Sequence Index')
plt.ylabel('Log-Likelihood')
plt.title('Per-Sequence Log-Likelihood')
plt.tight_layout()
plt.show()

## 3. Viterbi State Decoding

In [ ]:
from phase2_hmm.hmm_predict import predict_hidden_states, map_states_to_labels
from phase2_hmm.visualize import plot_state_sequence

# Decode states for the full concatenated sequence
all_obs = np.concatenate(obs_sequences)
all_states = predict_hidden_states(hmm_model, all_obs)
state_labels = map_states_to_labels(all_states)

print('State sequence (first 20):', state_labels[:20])

plot_state_sequence(
    all_states[:30],
    title='Viterbi Hidden State Sequence (first 30 events)',
    save_path=os.path.join(config.PLOTS_DIR, 'hmm_state_sequence.png')
)

## 4. Occurrence Probability Distribution

In [ ]:
from phase2_hmm.hmm_predict import compute_occurrence_probability

probabilities = [compute_occurrence_probability(hmm_model, seq) for seq in obs_sequences]
print(f'Probability range: [{min(probabilities):.4f}, {max(probabilities):.4f}]')
assert all(0.0 <= p <= 1.0 for p in probabilities), 'Probabilities out of [0,1] range!'
print('Probability range check: OK')

try:
    from phase2_hmm.visualize import plot_probability_distribution
    plot_probability_distribution(probabilities,
                                  save_path=os.path.join(config.PLOTS_DIR, 'hmm_probabilities.png'))
except Exception as e:
    print(f'KDE plot skipped: {e}')
    plt.hist(probabilities, bins=10, color='steelblue', edgecolor='white')
    plt.title('Occurrence Probability Distribution')
    plt.xlabel('Probability')
    plt.show()

## 5. Leave-One-Out Type Accuracy

In [ ]:
# Leave-one-out: train on N-1 sequences, predict held-out sequence's type
from phase2_hmm.hmm_model import build_hmm, train_hmm
from phase2_hmm.hmm_predict import predict_hidden_states

n = len(obs_sequences)
if n < 3:
    print('Too few sequences for LOO CV — skipping')
else:
    correct = 0
    for i in range(n):
        train_seqs = obs_sequences[:i] + obs_sequences[i+1:]
        train_lens = lengths[:i] + lengths[i+1:]
        test_seq   = obs_sequences[i]

        n_sym = len(dis_enc.classes_)
        loo_model = build_hmm(n_symbols=n_sym)
        import warnings; warnings.filterwarnings('ignore')
        loo_model = train_hmm(loo_model, train_seqs, train_lens)

        pred_states = predict_hidden_states(loo_model, test_seq)
        # 'Correct' if dominant predicted state matches dominant true state
        true_states = predict_hidden_states(hmm_model, test_seq)
        if pred_states[np.argmax(np.bincount(pred_states))] == true_states[np.argmax(np.bincount(true_states))]:
            correct += 1

    loo_acc = correct / n
    print(f'Leave-One-Out type accuracy: {loo_acc:.4f} ({correct}/{n})')

## 6. Classify Each Historical Event

In [ ]:
from phase2_hmm.hmm_predict import classify_landslide_type
import pandas as pd

rows = []
for i, row in df_clean.iterrows():
    disaster_desc = str(row.get('disaster_clean', 'Landslide'))
    location = str(row.get('location_clean', 'Unknown'))
    year = int(row.get('year_int', 2020))
    duration = int(row.get('duration_days', 1))

    result = classify_landslide_type(
        hmm_model, dis_enc,
        disaster_description=disaster_desc,
        location=location, duration=duration, year=year
    )
    rows.append({
        'location': location,
        'year': year,
        'actual_disaster': disaster_desc,
        'predicted_type': result['landslide_type'],
        'probability': result['occurrence_probability'],
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string())